# FoodWise — Recommendation Engine & Evaluation
This notebook demonstrates:
1. TF-IDF vectorization of restaurant content features.
2. Cosine similarity scoring.
3. Multi-factor personalized ranking (Content + Rating + Price + Delivery + Safety).
4. Automated, factual recommendation explanations.
5. "Find Similar Restaurants" content matching.
6. Offline benchmark of a Random baseline, a Top-rated (cuisine filter) baseline, Pure Content-Based, and the FoodWise Hybrid. There are no user interactions in the data, so this is not a measure of user satisfaction.

In [1]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np

sys.path.append("..")
from src.recommender import RestaurantRecommender
from src.evaluation import RecommendationEvaluator

recommender = RestaurantRecommender()
print(f"Recommender initialized with {len(recommender.df):,} restaurants across {len(recommender.cities)} cities.")

Recommender initialized with 44,708 restaurants across 97 cities.


## 1. Personalized Recommendations Demonstration
Query: A user in **Mumbai** looking for **Italian & Pizza**, budget under **₹250**, minimum rating **4.0**.

In [2]:
recommendations, meta = recommender.recommend(
    city="Mumbai",
    cuisines=["Italian", "Pizza"],
    budget_max=250,
    min_rating=4.0,
    top_k=5
)

print(f"Candidates Found: {meta['candidates_found']}")
for i, (_, row) in enumerate(recommendations.iterrows(), 1):
    print(f"{i}. {row['Restaurant Name']} | Score: {row['match_score']}% | ⭐ {row['Rating']} | ₹{int(row['price_numeric'])}")
    print(f"   Cuisines: {row['Cuisine']}")
    print(f"   Why: {row['explanation']}\n")

Candidates Found: 456
1. Shabbir's Tawakkal Sweets | Score: 59.5% | ⭐ 4.4 | ₹200
   Cuisines: Mithai, Fast Food, Sandwich, Rolls, Pizza
   Why: Cuisine and profile match · Rated 4.4/5 · ₹200 average price for one · 33 min listed delivery time

2. Ovenfresh | Score: 58.5% | ⭐ 4.4 | ₹150
   Cuisines: Italian, Pizza, Pasta, Mexican, Asian, Desserts, Beverages
   Why: Cuisine and profile match · Rated 4.4/5 · ₹150 average price for one · 27 min listed delivery time

3. Lil Gamby Pizza Shop | Score: 57.4% | ⭐ 4.2 | ₹200
   Cuisines: Pizza, Fast Food, Pasta, Italian
   Why: Cuisine and profile match · Rated 4.2/5 · ₹200 average price for one · 35 min listed delivery time

4. Le 15 Patisserie | Score: 54.0% | ⭐ 4.6 | ₹100
   Cuisines: Bakery, Desserts
   Why: Rated 4.6/5 · ₹100 average price for one · 22 min listed delivery time

5. Italian Bay | Score: 55.8% | ⭐ 4.1 | ₹150
   Cuisines: Italian, Pizza, Burger, Desserts, Pasta, Salad, Beverages
   Why: Cuisine and profile match · Rated 4.1/5 ·

## 2. "Find Similar Restaurants" Demonstration
Given a target restaurant, find other restaurants with highest TF-IDF content similarity.

In [3]:
sample_restaurant = recommendations.iloc[0]["Restaurant Name"]
print(f"Finding restaurants similar to: {sample_restaurant} in Mumbai")

similars = recommender.get_similar_restaurants(sample_restaurant, city="Mumbai", top_k=5)
for i, (_, s_row) in enumerate(similars.iterrows(), 1):
    print(f"{i}. {s_row['Restaurant Name']} | Similarity: {s_row['similarity_score']}% | Rating: {s_row['Rating']}")
    print(f"   Cuisines: {s_row['Cuisine']}\n")

Finding restaurants similar to: Shabbir's Tawakkal Sweets in Mumbai
1. Radha Krishna Sweets | Similarity: 54.1% | Rating: 3.9
   Cuisines: Mithai, Street Food, Sandwich, Rolls, Pizza, Fast Food, Sichuan

2. Cheezzila | Similarity: 41.9% | Rating: 3.9
   Cuisines: Pizza

3. Burrp Square Pizza | Similarity: 41.9% | Rating: 4.1
   Cuisines: Pizza

4. Balu's Classic | Similarity: 37.2% | Rating: 4.0
   Cuisines: Fast Food, Sandwich

5. Momos Magic | Similarity: 36.8% | Rating: 3.9
   Cuisines: Momos, Fast Food, Sandwich, Pizza



## 3. Recommendation System Evaluation Benchmark
Compare:
1. Random (eligible) baseline
2. Top-rated listings that match the cuisine
3. Pure Content-Based Filtering
4. FoodWise Hybrid Personalized Ranking

Cuisine Match only counts shared cuisine tags, so content-based methods score 1.0 by design. Compare the rating, price and diversity columns instead.

In [4]:
evaluator = RecommendationEvaluator(recommender)
benchmark_results = evaluator.run_benchmark(k=5)
print("=== RECOMMENDATION SYSTEM BENCHMARK ===")
benchmark_results

=== RECOMMENDATION SYSTEM BENCHMARK ===


,Model,Mean Cuisine Match@5,Mean Rating,Mean Price (Rs),Mean Diversity,Unique Items
0,Random (eligible),0.5,4.026,153.0,0.816,50
1,Top-rated (cuisine filter),1.0,4.472,151.0,0.726,50
2,Content-based,1.0,4.062,159.0,0.328,50
3,FoodWise hybrid,1.0,4.158,148.0,0.386,50
